# Version 1 — Réseau à une couche cachée
**Responsable : DIOP Alioune** · relecture : FALL Madjiguène

Parties 1.3, 1.4, 2 du sujet.

## 1. Imports

In [ ]:
import sys; sys.path.append("../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from har import config, data, models, training, metrics, plots, interpret
training.set_seed(0)
print("device :", training.DEVICE)

## 2. Chargement et exploration des données
(résumé ; l'exploration complète est dans `00_exploration.ipynb`)

In [ ]:
df = data.load_train()
df.shape, df[config.SUBJECT_COL].nunique()

## 3. Découpage apprentissage / validation et DataLoaders
Validation = 5 volontaires entiers (fenêtres consécutives qui se chevauchent → fuite si découpage par lignes).

In [ ]:
df = data.load_train()
df_tr, df_va, val_subjects = data.split_by_subject(df)
print("volontaires de validation :", val_subjects)
assert set(df_tr.subject) & set(df_va.subject) == set()
X_tr, y_tr, X_va, y_va, scaler = data.prepare(df_tr, df_va)
train_loader, val_loader = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=0)
IN_DIM = X_tr.shape[1]
X_tr.shape, X_va.shape

### 3.1 Découpage par lignes vs par volontaire (P1.3)

In [ ]:
from sklearn.linear_model import LogisticRegression
def logreg_val_score(df_tr, df_va):
    Xa, ya, Xb, yb, _ = data.prepare(df_tr, df_va)
    clf = LogisticRegression(max_iter=2000).fit(Xa, ya)
    return metrics.compute_metrics(yb, clf.predict(Xb))

r_rows = logreg_val_score(*data.split_random_rows(df))
r_subj = logreg_val_score(df_tr, df_va)
pd.DataFrame({"lignes aléatoires": r_rows, "par volontaire": r_subj})
# TODO : refaire avec le MLP 64 ; comparer au test en P5.2

### 3.2 Référence : régression logistique (P1.4)

In [ ]:
baseline = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
yp = baseline.predict(X_va)
print(metrics.compute_metrics(y_va, yp))
metrics.per_class_f1(y_va, yp)

## 4. Modèle et fonctions d'entraînement / évaluation
`train_epoch`, `evaluate`, `fit` sont dans `src/har/training.py`.

In [ ]:
model = models.MLP(IN_DIM, [64], activation="relu")
print(model, "\nparamètres :", models.count_params(model))

### 4.1 Vérifications (P2.1)
Perte initiale attendue : $-\ln(1/6) \approx 1{,}79$.

In [ ]:
crit = nn.CrossEntropyLoss()
print("perte initiale :", training.evaluate(model.to(training.DEVICE), val_loader, crit)["loss"], "attendu :", np.log(6))

# surapprentissage d'un petit lot
Xs, ys = next(iter(train_loader))
small = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xs, ys), batch_size=len(ys))
m = models.MLP(IN_DIM, [64]); opt = torch.optim.Adam(m.parameters(), 1e-2)
h = training.fit(m, small, small, opt, epochs=200)
print("perte finale sur le petit lot :", h["train_loss"][-1])

## 5. Boucle d'entraînement principale (P2.2)
### 5.1 Réglage du pas d'apprentissage

In [ ]:
# LR range test pour chaque optimiseur (à compléter par une petite grille log)
for opt_name in ["sgd", "sgd_momentum", "adam"]:
    training.set_seed(0)
    m = models.MLP(IN_DIM, [64])
    opt = training.make_optimizer(opt_name, m.parameters(), lr=1e-6)
    lrs, losses = training.lr_range_test(m, train_loader, opt)
    plt.semilogx(lrs, losses, label=opt_name)
plt.xlabel("lr"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3)

### 5.2 Activations × optimiseurs

In [ ]:
BEST_LR = {"sgd": 1e-1, "sgd_momentum": 1e-2, "adam": 1e-3}   # TODO : remplacer par les valeurs réglées
results = {}
for act in ["relu", "sigmoid", "tanh"]:
    for opt_name, lr in BEST_LR.items():
        training.set_seed(0)
        m = models.MLP(IN_DIM, [64], activation=act)
        opt = training.make_optimizer(opt_name, m.parameters(), lr=lr)
        results[(act, opt_name)] = training.fit(m, train_loader, val_loader, opt, epochs=50)
pd.DataFrame({k: {"val_acc": h["val_acc"][-1], "val_f1": h["val_f1"][-1]} for k, h in results.items()}).T

### 5.3 Variabilité sur plusieurs graines (P2.3)

In [ ]:
def build(act, opt_name, lr):
    def fn(seed):
        tl, vl = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=seed)
        m = models.MLP(IN_DIM, [64], activation=act)
        return m, tl, vl, training.make_optimizer(opt_name, m.parameters(), lr=lr)
    return fn

multi = {cfg: training.run_seeds(build(*cfg), seeds=config.SEEDS, epochs=50)
         for cfg in [("relu", "adam", 1e-3), ("tanh", "sgd_momentum", 1e-2)]}   # TODO : configs choisies
pd.DataFrame({str(k): {m: f"{v['summary'][m]['mean']:.4f} ± {v['summary'][m]['std']:.4f}"
                       for m in ("val_acc", "val_f1")} for k, v in multi.items()}).T

## 6. Visualisation et comparaison des résultats

In [ ]:
best = results[("relu", "adam")]   # TODO
plots.learning_curves(best, "MLP [64] ReLU Adam", save="v1_learning_curves")
plots.compare_curves({f"{a}/{o}": h for (a, o), h in results.items()}, key="val_f1", save="v1_compare")

**Accuracy vs F1 macro** : …  
**Convergence, stabilité, sous/sur-apprentissage** : …

## 7. Évaluation finale sur le jeu de test
⚠️ Exécutée **une seule fois**, quand la référence et la configuration V1 sont figées. Indépendant de la V2 : la comparaison globale se fait dans le rapport.

In [ ]:
# final_v1 = ...   # modèle MLP [64] retenu, réentraîné avec sa config figée
# df_test = data.load_test()
# X_te, y_te = data.to_xy(df_test); X_te = scaler.transform(X_te).astype(np.float32)
# test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.as_tensor(X_te), torch.as_tensor(y_te)), batch_size=512)
# res = {"référence (logreg)": metrics.compute_metrics(y_te, baseline.predict(X_te))}
# ev = training.evaluate(final_v1, test_loader, nn.CrossEntropyLoss(), return_preds=True)
# res["MLP [64]"] = {k: ev[k] for k in ("accuracy", "f1_macro")}
# display(pd.DataFrame(res).T)                                  # comparer aux scores de validation
# plots.confusion_heatmap(metrics.confusion(ev["y_true"], ev["y_pred"]), "V1 — test", save="v1_confusion_test")
# P1.3 : quel découpage de validation (lignes / volontaires) était le plus proche du test ?